In [8]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration

MODEL_ID = "openai/whisper-small"

device = "cuda" if torch.cuda.is_available() else "cpu"

processor = WhisperProcessor.from_pretrained(MODEL_ID)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)

model = model.to(device)

print(device)

Loading weights: 100%|██████████| 479/479 [00:00<00:00, 9560.77it/s]


cuda


In [9]:
import socket
import torch

print(socket.gethostname())
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

spark-f886
True
NVIDIA GB10


In [10]:
from pathlib import Path
import pandas as pd
import librosa

DATASET_DIR = Path("../../data/karakalpak-combined")

train_df = pd.read_csv(DATASET_DIR / "train.csv")

row = train_df.iloc[0]

audio_path = DATASET_DIR / row["audio_path"]
text = row["text"]

audio, sr = librosa.load(audio_path, sr=None)

print("Audio:", audio_path)
print("Sample rate:", sr)
print("Duration:", len(audio) / sr)
print("Text:", text)

Audio: ../../data/karakalpak-combined/train/audio/hf_audio_000001.wav
Sample rate: 16000
Duration: 6.95225
Text: kún uyasına sıymay dolanıp qıp qızıl shoqtay janıp batıwdıń az aq aldında tur


In [11]:
inputs = processor(
    audio,
    sampling_rate=sr,
    return_tensors="pt"
)

input_features = inputs.input_features

print(input_features.shape)

torch.Size([1, 80, 3000])


In [12]:
token_ids = processor.tokenizer(text).input_ids

decoded = processor.tokenizer.decode(
    token_ids,
    skip_special_tokens=True
)

print("Original:")
print(text)

print("\nDecoded:")
print(decoded)

print("\nToken IDs:")
print(token_ids)

Original:
kún uyasına sıymay dolanıp qıp qızıl shoqtay janıp batıwdıń az aq aldında tur

Decoded:
kún uyasına sıymay dolanıp qıp qızıl shoqtay janıp batıwdıń az aq aldında tur

Token IDs:
[50258, 50363, 74, 9453, 28266, 47973, 30201, 4199, 320, 360, 8658, 17091, 9505, 17091, 9505, 4041, 5492, 2223, 80, 83, 320, 25442, 17091, 7362, 699, 86, 26608, 5248, 7883, 257, 80, 16798, 8431, 3243, 50257]


In [13]:
processor.tokenizer.set_prefix_tokens(
    language="kk",
    task="transcribe"
)

In [14]:
from datasets import Dataset, DatasetDict
import pandas as pd

train_df = pd.read_csv(DATASET_DIR / "train.csv")
test_df = pd.read_csv(DATASET_DIR / "test.csv")

# Convert relative paths → full paths
train_df["audio_path"] = train_df["audio_path"].apply(
    lambda x: str(DATASET_DIR / x)
)

test_df["audio_path"] = test_df["audio_path"].apply(
    lambda x: str(DATASET_DIR / x)
)

train_dataset = Dataset.from_pandas(
    train_df,
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df,
    preserve_index=False
)

In [15]:
split = test_dataset.train_test_split(
    test_size=0.5,
    seed=42
)

dataset = DatasetDict({
    "train": train_dataset,
    "validation": split["train"],
    "test": split["test"],
})

dataset

DatasetDict({
    train: Dataset({
        features: ['audio_path', 'text', 'duration'],
        num_rows: 3708
    })
    validation: Dataset({
        features: ['audio_path', 'text', 'duration'],
        num_rows: 463
    })
    test: Dataset({
        features: ['audio_path', 'text', 'duration'],
        num_rows: 464
    })
})

In [16]:
dataset["train"][0]

{'audio_path': '../../data/karakalpak-combined/train/audio/hf_audio_000001.wav',
 'text': 'kún uyasına sıymay dolanıp qıp qızıl shoqtay janıp batıwdıń az aq aldında tur',
 'duration': 6.95225}

In [17]:
from pathlib import Path

sample = dataset["train"][0]

print(Path(sample["audio_path"]).exists())
print(sample["text"])
print(sample["duration"])

True
kún uyasına sıymay dolanıp qıp qızıl shoqtay janıp batıwdıń az aq aldında tur
6.95225


In [18]:
import librosa

def prepare_example(example):
    audio, sr = librosa.load(
        example["audio_path"],
        sr=16000
    )

    example["input_features"] = processor.feature_extractor(
        audio,
        sampling_rate=16000
    ).input_features[0]

    example["labels"] = processor.tokenizer(
        example["text"]
    ).input_ids

    return example

In [19]:
sample = dataset["train"][0]

prepared = prepare_example(sample)

print("Text:")
print(sample["text"])

print("\nInput feature shape:")
print(len(prepared["input_features"]), len(prepared["input_features"][0]))

print("\nLabel IDs:")
print(prepared["labels"])

Text:
kún uyasına sıymay dolanıp qıp qızıl shoqtay janıp batıwdıń az aq aldında tur

Input feature shape:
80 3000

Label IDs:
[50258, 50316, 50359, 50363, 74, 9453, 28266, 47973, 30201, 4199, 320, 360, 8658, 17091, 9505, 17091, 9505, 4041, 5492, 2223, 80, 83, 320, 25442, 17091, 7362, 699, 86, 26608, 5248, 7883, 257, 80, 16798, 8431, 3243, 50257]


In [20]:
print(
    processor.tokenizer.decode(
        prepared["labels"],
        skip_special_tokens=True
    )
)

kún uyasına sıymay dolanıp qıp qızıl shoqtay janıp batıwdıń az aq aldında tur


In [21]:
small_train = dataset["train"].select(range(50))

small_train = small_train.map(
    prepare_example
)

print(small_train)

Map: 100%|██████████| 50/50 [00:00<00:00, 110.50 examples/s]

Dataset({
    features: ['audio_path', 'text', 'duration', 'input_features', 'labels'],
    num_rows: 50
})


In [22]:
example = small_train[0]

print("Feature shape:")
print(len(example["input_features"]), len(example["input_features"][0]))

print("\nLabels:")
print(example["labels"])

print("\nDecoded:")
print(
    processor.tokenizer.decode(
        example["labels"],
        skip_special_tokens=True
    )
)

Feature shape:
80 3000

Labels:
[50258, 50316, 50359, 50363, 74, 9453, 28266, 47973, 30201, 4199, 320, 360, 8658, 17091, 9505, 17091, 9505, 4041, 5492, 2223, 80, 83, 320, 25442, 17091, 7362, 699, 86, 26608, 5248, 7883, 257, 80, 16798, 8431, 3243, 50257]

Decoded:
kún uyasına sıymay dolanıp qıp qızıl shoqtay janıp batıwdıń az aq aldında tur


In [23]:
from dataclasses import dataclass

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: any

    def __call__(self, features):
        input_features = [
            {"input_features": f["input_features"]}
            for f in features
        ]

        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        label_features = [
            {"input_ids": f["labels"]}
            for f in features
        ]

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["attention_mask"].ne(1),
            -100
        )

        batch["labels"] = labels

        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor
)



In [24]:
batch = data_collator([
    small_train[0],
    small_train[1]
])

print("Audio:")
print(batch["input_features"].shape)

print("\nLabels:")
print(batch["labels"].shape)

Audio:
torch.Size([2, 80, 3000])

Labels:
torch.Size([2, 37])


In [25]:
batch = {
    key: value.to(device)
    for key, value in batch.items()
}

model.train()

outputs = model(
    input_features=batch["input_features"],
    labels=batch["labels"]
)

print("Loss:", outputs.loss.item())
print("Logits shape:", outputs.logits.shape)

Loss: 5.240981578826904
Logits shape: torch.Size([2, 37, 51865])


In [27]:
from dataclasses import dataclass

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: any

    def __call__(self, features):
        input_features = [
            {"input_features": f["input_features"]}
            for f in features
        ]

        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        label_features = [
            {"input_ids": f["labels"]}
            for f in features
        ]

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["attention_mask"].ne(1),
            -100
        )

        # Remove BOS token if every label starts with it
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch

In [28]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor
)

In [29]:
batch = data_collator([
    small_train[0],
    small_train[1]
])

print(batch["input_features"].shape)
print(batch["labels"].shape)

torch.Size([2, 80, 3000])
torch.Size([2, 37])


In [30]:
from torch.utils.data import DataLoader

tiny_train = small_train.select(range(20))

train_loader = DataLoader(
    tiny_train,
    batch_size=1,
    shuffle=True,
    collate_fn=data_collator
)

In [31]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-5
)

In [36]:
model.train()

for epoch in range(2):
    total_loss = 0

    for batch in train_loader:
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        optimizer.zero_grad()

        outputs = model(
            input_features=batch["input_features"],
            labels=batch["labels"]
        )

        loss = outputs.loss

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(f"Epoch {epoch + 1}: loss = {avg_loss:.4f}")

Epoch 1: loss = 0.0009
Epoch 2: loss = 0.0009


In [37]:
sample = tiny_train[0]

input_features = torch.tensor(
    sample["input_features"],
    dtype=torch.float32
).unsqueeze(0).to(device)

model.eval()

with torch.no_grad():
    predicted_ids = model.generate(
        input_features,
        language="kk",
        task="transcribe",
        max_new_tokens=128
    )

prediction = processor.tokenizer.decode(
    predicted_ids[0],
    skip_special_tokens=True
)

print("TARGET:")
print(sample["text"])

print("\nPREDICTION:")
print(prediction)

[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TARGET:
kún uyasına sıymay dolanıp qıp qızıl shoqtay janıp batıwdıń az aq aldında tur

PREDICTION:
kún uyasına sıymay dolanıp qıp qızıl shoqtay janıp batıwdıń az aq aldında tur


In [ ]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-small"
).to(device)

model.config.use_cache = False

In [39]:
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import soundfile as sf
import torch

from torch.utils.data import Dataset, DataLoader
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
)


# ============================================================
# 1. SETTINGS
# ============================================================

MODEL_ID = "openai/whisper-small"
LANGUAGE = "kk"
TASK = "transcribe"
SAMPLE_RATE = 16000
BATCH_SIZE = 1
SEED = 42


# ============================================================
# 2. DEVICE
# ============================================================

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device:", device)


# ============================================================
# 3. FIND DATASET AUTOMATICALLY
# ============================================================

possible_paths = [
    Path("data/karakalpak-combined"),
    Path("../data/karakalpak-combined"),
    Path("../../data/karakalpak-combined"),
]

DATASET_DIR = next(
    (path.resolve() for path in possible_paths if path.exists()),
    None
)

if DATASET_DIR is None:
    raise FileNotFoundError(
        "Could not find data/karakalpak-combined"
    )

print("Dataset:", DATASET_DIR)


# ============================================================
# 4. LOAD CSV FILES
# ============================================================

train_df = pd.read_csv(DATASET_DIR / "train.csv")
test_df = pd.read_csv(DATASET_DIR / "test.csv")

print("Original train:", len(train_df))
print("Original test:", len(test_df))


# ============================================================
# 5. CREATE VALIDATION SPLIT
#    Keep original test set untouched.
# ============================================================

rng = np.random.default_rng(SEED)

indices = rng.permutation(len(train_df))

validation_size = int(len(train_df) * 0.10)

validation_indices = indices[:validation_size]
train_indices = indices[validation_size:]

validation_df = train_df.iloc[validation_indices].reset_index(drop=True)
train_df = train_df.iloc[train_indices].reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("\nFinal splits:")
print("Train:", len(train_df))
print("Validation:", len(validation_df))
print("Test:", len(test_df))


# ============================================================
# 6. LOAD WHISPER SMALL
# ============================================================

processor = WhisperProcessor.from_pretrained(
    MODEL_ID
)

processor.tokenizer.set_prefix_tokens(
    language=LANGUAGE,
    task=TASK
)

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID
)

model.config.use_cache = False

model.generation_config.language = LANGUAGE
model.generation_config.task = TASK

model = model.to(device)

print("\nWhisper loaded:", MODEL_ID)


# ============================================================
# 7. PYTORCH DATASET
# ============================================================

class KarakalpakDataset(Dataset):
    def __init__(self, dataframe, dataset_dir):
        self.dataframe = dataframe
        self.dataset_dir = dataset_dir

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        return {
            "audio_path": str(
                self.dataset_dir / row["audio_path"]
            ),
            "text": str(row["text"]),
            "duration": float(row["duration"]),
        }


train_dataset = KarakalpakDataset(
    train_df,
    DATASET_DIR
)

validation_dataset = KarakalpakDataset(
    validation_df,
    DATASET_DIR
)

test_dataset = KarakalpakDataset(
    test_df,
    DATASET_DIR
)


# ============================================================
# 8. DATA COLLATOR
#    WAV -> Whisper features
#    Karakalpak text -> Whisper labels
# ============================================================

@dataclass
class WhisperDataCollator:
    processor: object
    decoder_start_token_id: int

    def __call__(self, features):

        audios = []
        texts = []

        for feature in features:

            audio, sample_rate = sf.read(
                feature["audio_path"],
                dtype="float32"
            )

            # Dataset should already be clean.
            # We verify instead of silently modifying it.
            if sample_rate != SAMPLE_RATE:
                raise ValueError(
                    f"Expected 16000 Hz, got {sample_rate}: "
                    f"{feature['audio_path']}"
                )

            if audio.ndim != 1:
                raise ValueError(
                    f"Expected mono audio: "
                    f"{feature['audio_path']}"
                )

            audios.append(audio)
            texts.append(feature["text"])

        # ----------------------------------------------------
        # AUDIO -> LOG-MEL FEATURES
        # ----------------------------------------------------

        batch = self.processor.feature_extractor(
            audios,
            sampling_rate=SAMPLE_RATE,
            return_tensors="pt"
        )

        # ----------------------------------------------------
        # KARAKALPAK TEXT -> TOKEN IDS
        # ----------------------------------------------------

        labels_batch = self.processor.tokenizer(
            texts,
            padding=True,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"]

        # Ignore padding when calculating loss
        labels = labels.masked_fill(
            labels_batch["attention_mask"].ne(1),
            -100
        )

        # Remove decoder start token if tokenizer added it
        if (
            labels.shape[1] > 0
            and
            (labels[:, 0] == self.decoder_start_token_id)
            .all()
            .item()
        ):
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch


data_collator = WhisperDataCollator(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id
)


# ============================================================
# 9. DATALOADERS
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator,
    num_workers=0,
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
    num_workers=0,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
    num_workers=0,
)


# ============================================================
# 10. TEST ONE REAL BATCH
# ============================================================

batch = next(iter(train_loader))

print("\nInput features:")
print(batch["input_features"].shape)

print("\nLabels:")
print(batch["labels"].shape)

print("\nExample transcript:")
print(train_dataset[0]["text"])

print("\nEverything is ready.")

Device: cuda
Dataset: /home/intern/AI & ML/data/karakalpak-combined
Original train: 3708
Original test: 927

Final splits:
Train: 3338
Validation: 370
Test: 927


Loading weights: 100%|██████████| 479/479 [00:00<00:00, 6403.84it/s]



Whisper loaded: openai/whisper-small

Input features:
torch.Size([1, 80, 3000])

Labels:
torch.Size([1, 64])

Example transcript:
al eger sen hár kúni hár waqıtta keletuǵın bolsań men ózimniń júregimdi qaysı saatqa tayarlawdı bilmeymen

Everything is ready.


In [40]:
import os
import math
import torch
from pathlib import Path
from tqdm.auto import tqdm

# ============================================================
# TRAINING SETTINGS
# ============================================================

EPOCHS = 3
LEARNING_RATE = 1e-5

# batch_size is already 1 in your DataLoader
GRAD_ACCUM_STEPS = 8

MAX_GRAD_NORM = 1.0

CHECKPOINT_DIR = Path("../../checkpoints/whisper-small-kaa")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# MEMORY SAVING
# ============================================================

model.gradient_checkpointing_enable()
model.config.use_cache = False

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE
)


# ============================================================
# VALIDATION FUNCTION
# ============================================================

@torch.no_grad()
def validate():
    model.eval()

    total_loss = 0.0
    total_batches = 0

    for batch in tqdm(
        validation_loader,
        desc="Validation",
        leave=False
    ):
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        outputs = model(
            input_features=batch["input_features"],
            labels=batch["labels"]
        )

        total_loss += outputs.loss.item()
        total_batches += 1

    model.train()

    return total_loss / total_batches


# ============================================================
# TRAINING
# ============================================================

best_validation_loss = float("inf")

model.train()

for epoch in range(EPOCHS):

    print(f"\n========== EPOCH {epoch + 1}/{EPOCHS} ==========")

    optimizer.zero_grad()

    total_loss = 0.0
    step_count = 0

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}"
    )

    for step, batch in enumerate(progress_bar):

        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        outputs = model(
            input_features=batch["input_features"],
            labels=batch["labels"]
        )

        loss = outputs.loss

        # divide because we accumulate gradients
        scaled_loss = loss / GRAD_ACCUM_STEPS

        scaled_loss.backward()

        total_loss += loss.item()
        step_count += 1

        # update weights every 8 samples
        if (
            (step + 1) % GRAD_ACCUM_STEPS == 0
            or
            step + 1 == len(train_loader)
        ):

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                MAX_GRAD_NORM
            )

            optimizer.step()
            optimizer.zero_grad()

        avg_loss = total_loss / step_count

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}",
            avg=f"{avg_loss:.4f}"
        )


    # ========================================================
    # VALIDATION
    # ========================================================

    train_loss = total_loss / step_count

    validation_loss = validate()

    print(
        f"\nEpoch {epoch + 1} finished"
    )

    print(
        f"Train loss:      {train_loss:.4f}"
    )

    print(
        f"Validation loss: {validation_loss:.4f}"
    )


    # ========================================================
    # SAVE BEST MODEL
    # ========================================================

    if validation_loss < best_validation_loss:

        best_validation_loss = validation_loss

        save_path = CHECKPOINT_DIR / "best"

        model.save_pretrained(save_path)
        processor.save_pretrained(save_path)

        print(
            f"Best model saved to: {save_path}"
        )


    # Free some cached memory
    if device.type == "mps":
        torch.mps.empty_cache()


print("\nTraining finished.")
print(
    "Best validation loss:",
    best_validation_loss
)


========== EPOCH 1/3 ==========


Epoch 1: 100%|██████████| 3338/3338 [13:23<00:00,  4.15it/s, avg=0.7489, loss=0.0713]



Epoch 1 finished
Train loss:      0.7489
Validation loss: 0.4988


Writing model shards: 100%|██████████| 1/1 [00:08<00:00,  8.65s/it]


Best model saved to: ../../checkpoints/whisper-small-kaa/best

========== EPOCH 2/3 ==========


...

Best model saved to: ../../checkpoints/whisper-small-kaa/best

========== EPOCH 3/3 ==========


Epoch 3: 100%|██████████| 3338/3338 [12:22<00:00,  4.50it/s, avg=0.1236, loss=0.0710]



Epoch 3 finished
Train loss:      0.1236
Validation loss: 0.4149


Writing model shards: 100%|██████████| 1/1 [00:08<00:00,  8.12s/it]

Best model saved to: ../../checkpoints/whisper-small-kaa/best

Training finished.
Best validation loss: 0.41486456200253924


In [41]:
import torch
import soundfile as sf
from jiwer import wer, cer
from tqdm.auto import tqdm

@torch.no_grad()
def evaluate_stt(model, dataset, limit=None):
    model.eval()

    predictions = []
    references = []

    n = len(dataset) if limit is None else min(limit, len(dataset))

    for i in tqdm(range(n), desc="WER/CER evaluation"):
        sample = dataset[i]

        audio, sr = sf.read(
            sample["audio_path"],
            dtype="float32"
        )

        inputs = processor.feature_extractor(
            audio,
            sampling_rate=16000,
            return_tensors="pt"
        )

        input_features = inputs.input_features.to(device)

        predicted_ids = model.generate(
            input_features,
            language="kk",
            task="transcribe",
            max_new_tokens=128
        )

        prediction = processor.tokenizer.decode(
            predicted_ids[0],
            skip_special_tokens=True
        ).strip()

        reference = sample["text"].strip()

        predictions.append(prediction)
        references.append(reference)

    model.train()

    return {
        "WER": wer(references, predictions),
        "CER": cer(references, predictions),
        "predictions": predictions,
        "references": references,
    }

In [43]:
results = evaluate_stt(
    model,
    validation_dataset,
    limit=20
)

print("WER:", results["WER"]*100)
print("CER:", results["CER"]*100)

WER/CER evaluation: 100%|██████████| 20/20 [00:05<00:00,  3.50it/s]

WER: 34.285714285714285
CER: 7.387706855791961


In [44]:
for ref, pred in zip(
    results["references"][:5],
    results["predictions"][:5]
):
    print("REFERENCE :", ref)
    print("PREDICTION:", pred)
    print()
    

REFERENCE : adamlar opırıq tisin pútinlewge altın taba almay júrgende qabaq soǵıwǵa jetkendey altındı qayaqtan tabadı
PREDICTION: adamlar opırıq tısın pútinlewge altın taba alma júrgende qabaq soǵıwǵa jetkendey altındı qayaqtıń taba da

REFERENCE : jaqtı dúnyadan jurt qatarı payımdı almay kete beremen be
PREDICTION: jaqtı dúnyadan jurt qatarı payımdı almay kete biremen be

REFERENCE : ol basqa sóyley almay tıǵılıp qaldı
PREDICTION: ol basqa sóyley almay tıǵılıp qaldı

REFERENCE : bilmeymen dep qaytaladı geograf
PREDICTION: bilmeymen dep qaytaladı geograf

REFERENCE : endi kete ber dedi kishkene shahzada
PREDICTION: endi kete ber dedi kishkene shahzada

